# Create Embeddings with CNN embed_150

In this notebook, we use dataset created by @ummaymariamuna and embed all the videos from that dataset using custom CNN model.

Dataset Structure:

```
/train
|- anomalous
|- nonanomalous
```



In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

In [ ]:
# Set the device to use (e.g., 'cpu', 'cuda', 'mps')
device = torch.device("mps" if torch.backends.mps.is_available() else
                      ("cuda" if torch.cuda.is_available() else "cpu"))

# Select Device According to Availability
print("Device selected:", device)

# If the device is CUDA, print the device capability
if device.type == "cuda":
    os.system("nvidia-smi")
    print()
    print("Device type:", device.type)
    print("Capability:", torch.cuda.get_device_capability(device))
else:
    print("Device capabilities are limited on MPSs and CPUs.")

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1).to(device)
        self.conv2 = nn.Conv2d(64, 128, 3, 1, 1).to(device)
        self.conv3 = nn.Conv2d(128, 256, 3, 1, 1).to(device)
        self.pool = nn.MaxPool2d(2, 2).to(device)
        self.fc1 = nn.Linear(256 * 28 * 28, 4096).to(device)
        self.fc2 = nn.Linear(4096, 2048).to(device)
        self.fc3 = nn.Linear(4096, 1024).to(device)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 256 * 28 * 28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return x
    
model = CNN()
model.to(device)
model.eval()

In [ ]:
def get_frames(path):
    video = cv2.VideoCapture(path)
    if not video.isOpened():
        raise IOError("Error reading video file")

    frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

    frames = []
    for i in range(0, frame_count, 2):
        video.set(1, i)
        ret, frame = video.read()
        if ret:
            frame = cv2.resize(frame, (224, 224))
            frame = frame.astype("float32") / 255.0
            frames.append(frame)
    video.release()
    return np.array(frames)

In [ ]:
import numpy as np
import glob
from random import shuffle
import tqdm

train_path = '/kaggle/input/augmenteddataset/train/*/*.mp4'
train_paths = glob.glob(train_path)
train_labels = [1 if i.split('/')[5] == 'anomalous' else 0 for i in train_paths]

train_zip = list(zip(train_paths, train_labels))
shuffle(train_zip)
train_paths, train_labels = zip(*train_zip)

train_labels = list(train_labels)

In [ ]:
def pad(x):
    assert len(x.shape) == 4
    
    max_frames = 300 // 2
    max_height = 224
    max_width = 224
    max_channel = 3
    
    padded_x = []
    
    pad_widths = [(0, max_frames - x.shape[0]),
                  (0, max_height - x.shape[1]),
                  (0, max_width - x.shape[2]),
                  (0, max_channel - x.shape[3])]
    
    padded_x = np.pad(x, pad_widths, mode='constant', constant_values=0.0)
    
    del pad_widths
    
    return padded_x
    

In [ ]:
from tqdm import tqdm

video_embeddings = []
video_labels = []

# test_path = train_paths[:20]
# test_label = train_labels[:20]

for path, label in zip(tqdm(train_paths), train_labels):
    try:
        frames = get_frames(path)

        # check the shape
        if frames.shape[0] != 150 or frames.shape[1] != 224 or frames.shape[2] != 224 or frames.shape[3] !=3:
            print(f"Shape mismatch for video {path}")
            print(f"Found Shape: {frames.shape}")
            print("Fixing shape by Padding")
            frames = pad(frames)
            print(f"Fixed Shape: {frames.shape}")

        # Reshaping the frame for CNN input
        frames = torch.tensor(frames).permute(0, 3, 1, 2).to(device)

        # Get embeddings for each frame
        embeddings_list = []
        for frame in frames:
            frame = frame.unsqueeze(0)
            frame.to(device)
            embeddings = model(frame)
            embeddings_list.append(embeddings.flatten().cpu().detach().numpy())

        embeddings_list = np.asarray(embeddings_list)    
        video_embeddings.append(embeddings_list)
        video_labels.append(label)
    except IOError as e:
        print(f"Skipping Video file {path} and label {label}")
    
video_embeddings = np.asarray(video_embeddings)
video_labels = np.asarray(video_labels)

print(video_embeddings.shape, video_labels.shape)

np.save('/kaggle/working/train_embeddings.npy', video_embeddings) 
np.save('/kaggle/working/train_labels.npy', video_labels)

In [ ]:
x = np.load('/kaggle/working/train_embeddings.npy')
print(x.shape)
y = np.load('/kaggle/working/train_labels.npy')
print(y.shape)